# ПЗ 6 — Классификация ResNet

**Задача:** читать кадры из Drive, классифицировать через ResNet50 (ImageNet), сохранить топ-5 классов в Drive.

**Стек:** `torch`, `torchvision`, `Pillow`, `pandas`

In [ ]:
!pip install torch torchvision Pillow tqdm -q

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive', force_remount=True)

BASE_DRIVE  = '/content/drive/MyDrive/cv-frames'
FRAMES_DIR  = f'{BASE_DRIVE}/кадры'
RESULTS_DIR = f'{BASE_DRIVE}/результаты'

os.makedirs(RESULTS_DIR, exist_ok=True)

frames = sorted(f for f in os.listdir(FRAMES_DIR) if f.endswith('.jpg'))
print(f'✅ Найдено кадров: {len(frames)}')

In [ ]:
import json, torch, urllib.request
import pandas as pd
from PIL import Image
from torchvision import models, transforms
from tqdm.notebook import tqdm

# Метки ImageNet
urllib.request.urlretrieve(
    'https://raw.githubusercontent.com/anishathalye/imagenet-simple-labels/master/imagenet-simple-labels.json',
    '/content/imagenet_labels.json'
)
with open('/content/imagenet_labels.json') as f:
    LABELS = json.load(f)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1).to(DEVICE).eval()
print(f'✅ ResNet50 загружен | {DEVICE}')

In [ ]:
transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

results = []
with torch.no_grad():
    for fname in tqdm(frames, desc='ResNet'):
        img = Image.open(f'{FRAMES_DIR}/{fname}').convert('RGB')
        inp = transform(img).unsqueeze(0).to(DEVICE)
        probs = torch.softmax(model(inp), dim=1)[0]
        top5 = torch.topk(probs, 5)
        for rank, (idx, prob) in enumerate(zip(top5.indices.tolist(), top5.values.tolist()), 1):
            results.append({
                'frame': fname, 'rank': rank,
                'class_name': LABELS[idx],
                'probability': round(prob, 4),
            })

df = pd.DataFrame(results)
OUT = f'{RESULTS_DIR}/resnet_classifications.csv'
df.to_csv(OUT, index=False)
print(f'✅ Классифицировано кадров: {len(frames)} → {OUT}')
df[df.rank==1].head(10)